# NB_lab_test_trends — Silver to Gold

Builds `adb_caresync.gold.lab_test_trends` (one row per test type per month) per the STTM `lab_test_trends` sheet: lab_results aggregated by test_name, year and month, LEFT JOIN the calendar dimension on the first day of month, with a LAG() month-over-month abnormal-rate trend. Full rebuild.

Notes: the calendar dimension is `adb_caresync.silver.calender` (project spelling). Silver `lab_results` contains duplicate `lab_result_id` rows, so it is de-duplicated (latest `update_timestamp`).

In [0]:
%sql
-- Gold: lab_test_trends | Grain: one row per test_name per month | Load: FULL rebuild
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends
COMMENT 'Monthly lab test volumes and abnormal/critical rates per test type with month-over-month abnormal-rate trend. One row per test_name per month.'
AS
WITH lab_results AS (
  SELECT * FROM adb_caresync.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    year(test_date)                             AS year,
    month(test_date)                            AS month,
    COUNT(*)                                    AS total_tests,
    COUNT_IF(result_status = 'Abnormal')        AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')        AS critical_test_count
  FROM lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(round(abnormal_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(round(critical_test_count * 100.0 / total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
trend AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  t.test_name                                                           AS test_name,
  CAST(t.year AS INT)                                                   AS year,
  CAST(t.month AS INT)                                                  AS month,
  c.month_name                                                          AS month_name,
  c.quarter_name                                                        AS quarter_name,
  CAST(t.total_tests AS INT)                                            AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                    AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                    AS critical_test_count,
  t.abnormal_rate_pct                                                   AS abnormal_rate_pct,
  t.critical_rate_pct                                                   AS critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                  AS prev_month_abnormal_rate_pct,
  CASE WHEN t.prev_month_abnormal_rate_pct IS NULL THEN NULL
       ELSE CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                                                   AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN adb_caresync.silver.calender c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT);

In [0]:
%python
# Return the number of records written to gold.lab_test_trends (full rebuild = total rows in table)
# ADF: read via @activity('<NotebookActivity>').output.runOutput
target_table = "adb_caresync.gold.lab_test_trends"
record_count = spark.table(target_table).count()
print(f"FULL load completed for {target_table}. record_count = {record_count}")
dbutils.notebook.exit(str(record_count))